In [ ]:
import numpy as np
import pandas as pd

In [ ]:
# Set random seed for reproducibility
np.random.seed(42)

# Generate synthetic data
n_samples = 1000

# Features
size_sqm = np.random.randint(50, 300, n_samples)
bedrooms = np.random.randint(1, 5, n_samples)
age = np.random.randint(0, 50, n_samples)

# Create a DataFrame
data = pd.DataFrame({
    'size_sqm': size_sqm,
    'bedrooms': bedrooms,
    'age': age
})

# Generate target variable (house price in 1000's of USD)
#  This is a simplified model, you can adjust the coefficients and noise for more realistic data
price = 50 + 10 * data['size_sqm'] + 5 * data['bedrooms'] - 2 * data['age'] + np.random.normal(0, 20, n_samples)
data['price'] = price

# Print the first few rows of the dataset
print(data)

     size_sqm  bedrooms  age        price
0         152         4    6  1563.016415
1         229         3   13  2333.506855
2         142         1   14  1436.450614
3          64         3    6   700.233532
4         156         1    8  1609.074721
..        ...       ...  ...          ...
995       297         1   43  2887.832632
996        82         3    0   870.369732
997       167         2   29  1651.536090
998       114         1   20  1134.401307
999       195         2   19  1957.019371

[1000 rows x 4 columns]


# Vectorization

Vectorization in ML programming is the process of optimizing code to perform operations on entire vectors or matrices at once, rather than using explicit loops.

Benefits:
- Significantly faster execution (takes advantage of SIMD instructions)
- More concise and readable code
- Better utilization of modern CPU/GPU architectures
- Improved scalability for large datasets

## Linear Regression without Vectorization

In [ ]:
def model(x, w, b):
    """
    Compute the linear regression model.

    :param x: List of feature values for a single instance
    :param w: List of weights
    :param b: Bias term
    :return: Predicted value
    """
    result = 0
    for i in range(len(x)):
        result += w[i] * x[i]
    result += b
    return result

def loss(y, y_hat):
    """
    Compute the Mean Squared Error loss.

    :param y: List of true values
    :param y_hat: List of predicted values
    :return: MSE loss
    """
    total_error = 0

    m = len(y)
    for i in range(m):
        total_error += (y[i] - y_hat[i]) ** 2
    return total_error / (2 * m)

def optimize(X, y, n_iter, alpha):
    """
    Perform gradient descent optimization.

    :param X: List of lists, where each inner list is a feature vector
    :param y: List of true output values
    :param n_iter: Number of iterations for gradient descent
    :param alpha: Learning rate
    :return: Optimized weights and bias
    """
    # Initialize weights and bias
    n_features = len(X[0])
    w = [0.0] * n_features
    b = 0.0

    for iteration in range(n_iter):
        # Compute predictions
        y_hat = []
        for i in range(len(X)):
            y_hat.append(model(X[i], w, b))

        # Compute gradients
        dw = [0.0] * n_features
        db = 0.0
        for i in range(len(X)):
            for j in range(n_features):
                dw[j] += (y_hat[i] - y[i]) * X[i][j]
            db += (y_hat[i] - y[i])

        # Update weights and bias
        for j in range(n_features):
            w[j] -= alpha * dw[j] / len(X)
        b -= alpha * db / len(X)

        # Print loss every 100 iterations
        if (iteration + 1) % 100 == 0:
            current_loss = loss(y, y_hat)
            print(f"Iteration {iteration + 1}, Loss: {current_loss}")

    return w, b

In [ ]:
# Load X and y from the house price dataset
X = data[['size_sqm', 'bedrooms', 'age']].values
y = data['price'].values

# Convert X and y to lists
X = X.tolist()
y = y.tolist()

# Normalize Xs
X = (X - np.mean(X, axis=0)) / np.std(X, axis=0)

n_iter = 1000
alpha = 0.01

%timeit w, b = optimize(X, y, n_iter, alpha)
w, b = optimize(X, y, n_iter, alpha)
print(f"Final weights: {w}")
print(f"Final bias: {b}")

# Make a prediction for a new house (size_sqm, bedrooms, age)
new_x = [84, 2, 35]
new_x = (new_x - np.mean(X, axis=0)) / np.std(X, axis=0)

prediction = model(new_x, w, b)
print(f"Prediction for {new_x}: {prediction}")

Iteration 100, Loss: 255979.92905157167
Iteration 200, Loss: 34501.57835949038
Iteration 300, Loss: 4800.305533443483
Iteration 400, Loss: 812.9224376517906
Iteration 500, Loss: 277.02671362389026
Iteration 600, Loss: 204.92182764301862
Iteration 700, Loss: 195.20876592224548
Iteration 800, Loss: 193.89876402662878
Iteration 900, Loss: 193.72186252225754
Iteration 1000, Loss: 193.69794277378023
Iteration 100, Loss: 255979.92905157167
Iteration 200, Loss: 34501.57835949038
Iteration 300, Loss: 4800.305533443483
Iteration 400, Loss: 812.9224376517906
Iteration 500, Loss: 277.02671362389026
Iteration 600, Loss: 204.92182764301862
Iteration 700, Loss: 195.20876592224548
Iteration 800, Loss: 193.89876402662878
Iteration 900, Loss: 193.72186252225754
Iteration 1000, Loss: 193.69794277378023
Iteration 100, Loss: 255979.92905157167
Iteration 200, Loss: 34501.57835949038
Iteration 300, Loss: 4800.305533443483
Iteration 400, Loss: 812.9224376517906
Iteration 500, Loss: 277.02671362389026
Iterati

## Linear Regression with Vectorization

In [ ]:
import numpy as np

def model(x, w, b):
    """
    Compute the linear regression model.

    :param x: NumPy array of feature values for a single instance
    :param w: NumPy array of weights
    :param b: Bias term (scalar)
    :return: Predicted value
    """
    return np.dot(x, w) + b

def loss(y, y_hat):
    """
    Compute the Mean Squared Error loss.

    :param y: NumPy array of true values
    :param y_hat: NumPy array of predicted values
    :return: MSE loss
    """
    return np.mean((y - y_hat) ** 2) * 0.5

def optimize(X, y, n_iter, alpha):
    """
    Perform gradient descent optimization.

    :param X: NumPy 2D array, where each row is a feature vector
    :param y: NumPy 1D array of true output values
    :param n_iter: Number of iterations for gradient descent
    :param alpha: Learning rate
    :return: Optimized weights and bias
    """
    # Initialize weights and bias
    n_features = X.shape[1]
    w = np.zeros(n_features)
    b = 0.0

    for iteration in range(n_iter):
        # Compute predictions
        y_hat = model(X, w, b)

        # Compute gradients
        dw = np.dot(X.T, (y_hat - y)) / len(y)
        db = np.mean(y_hat - y)

        # Update weights and bias
        w -= alpha * dw
        b -= alpha * db

        # Print loss every 100 iterations
        if (iteration + 1) % 100 == 0:
            current_loss = loss(y, y_hat)
            print(f"Iteration {iteration + 1}, Loss: {current_loss}")

    return w, b

In [ ]:
# Load X and y from the house price dataset
X = data[['size_sqm', 'bedrooms', 'age']].values
y = data['price'].values

# Convert X and y to lists
X = X.tolist()
y = y.tolist()

# Normalize Xs
X = (X - np.mean(X, axis=0)) / np.std(X, axis=0)

n_iter = 1000
alpha = 0.01

# Measure training time
%timeit w, b = optimize(X, y, n_iter, alpha)
w, b = optimize(X, y, n_iter, alpha)

print(f"Final weights: {w}")
print(f"Final bias: {b}")

# Make a prediction for a new house (size_sqm, bedrooms, age)
new_x = np.array([84, 2, 35])
new_x = (new_x - np.mean(X, axis=0)) / np.std(X, axis=0)

prediction = model(new_x, w, b)
print(f"Prediction for {new_x}: {prediction}")

Iteration 100, Loss: 255979.9290515717
Iteration 200, Loss: 34501.578359490304
Iteration 300, Loss: 4800.305533443476
Iteration 400, Loss: 812.9224376517911
Iteration 500, Loss: 277.02671362389
Iteration 600, Loss: 204.92182764301867
Iteration 700, Loss: 195.2087659222457
Iteration 800, Loss: 193.89876402662887
Iteration 900, Loss: 193.72186252225725
Iteration 1000, Loss: 193.69794277378008
Iteration 100, Loss: 255979.9290515717
Iteration 200, Loss: 34501.578359490304
Iteration 300, Loss: 4800.305533443476
Iteration 400, Loss: 812.9224376517911
Iteration 500, Loss: 277.02671362389
Iteration 600, Loss: 204.92182764301867
Iteration 700, Loss: 195.2087659222457
Iteration 800, Loss: 193.89876402662887
Iteration 900, Loss: 193.72186252225725
Iteration 1000, Loss: 193.69794277378008
Iteration 100, Loss: 255979.9290515717
Iteration 200, Loss: 34501.578359490304
Iteration 300, Loss: 4800.305533443476
Iteration 400, Loss: 812.9224376517911
Iteration 500, Loss: 277.02671362389
Iteration 600, Los